# 🎯 Job Recommendation System
### Resume Text → Job Title Prediction | TF-IDF + Cosine Similarity

**Dataset:** `kandij/job-recommendation-datasets`  
**Files used:** `Experience Data.csv`, `jobs2.csv`, `Salary Data.csv`, `combo_data.csv`  
**Run on:** Kaggle Notebooks (GPU not required)

---
**Pipeline:**
1. Load all 4 CSV files from `/kaggle/input/`
2. Inspect columns → extract `text` + `job_title` per file
3. Merge into one unified corpus
4. Clean & preprocess text
5. TF-IDF Vectorization → Save `.pkl` to `/kaggle/working/`
6. Cosine Similarity engine
7. Interactive resume input → Top-N Job Title predictions

## 📦 Step 1: Imports & Setup

In [ ]:
import os
import re
import glob
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import defaultdict, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

# ── Paths ──────────────────────────────────────────────────────
INPUT_DIR  = '/kaggle/input/job-recommendation-datasets/'
OUTPUT_DIR = '/kaggle/working/'
MODEL_DIR  = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

print('Input  dir:', INPUT_DIR)
print('Model  dir:', MODEL_DIR)
print('\nFiles in dataset:')
for f in sorted(glob.glob(INPUT_DIR + '**/*', recursive=True)):
    if os.path.isfile(f):
        print(f'  {os.path.basename(f):40s}  {os.path.getsize(f)/1024:.1f} KB')

## 📁 Step 2: Load All 4 CSV Files

In [ ]:
def safe_read(path):
    """Read CSV with fallback encoding."""
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            df = pd.read_csv(path, encoding=enc, on_bad_lines='skip')
            print(f'  ✅ Loaded [{enc}]: {os.path.basename(path):35s} → shape {df.shape}')
            print(f'     Columns: {df.columns.tolist()}')
            return df
        except Exception:
            continue
    print(f'  ❌ Could not read: {path}')
    return None

# ── Known file names in this dataset ───────────────────────────
FILE_NAMES = [
    'Experience Data.csv',
    'jobs2.csv',
    'Salary Data.csv',
    'combo_data.csv',
]

raw = {}
print('Loading files...\n')
for fname in FILE_NAMES:
    fpath = os.path.join(INPUT_DIR, fname)
    if os.path.exists(fpath):
        raw[fname] = safe_read(fpath)
    else:
        # fallback: search recursively
        matches = glob.glob(INPUT_DIR + f'**/{fname}', recursive=True)
        if matches:
            raw[fname] = safe_read(matches[0])
        else:
            print(f'  ⚠️  Not found: {fname}')

print(f'\nLoaded {len(raw)} file(s)')

## 🔎 Step 3: Inspect Each File in Detail

In [ ]:
for fname, df in raw.items():
    if df is None:
        continue
    print(f'\n{'─'*60}')
    print(f'📄  {fname}')
    print(f'    Shape   : {df.shape}')
    print(f'    Columns : {df.columns.tolist()}')
    print(f'    Nulls   : {df.isnull().sum().to_dict()}')
    display(df.head(2))

## 🔗 Step 4: Extract & Merge — Build Unified Corpus

Each file may have different column names. We map each to a standard:
- `resume_text` — the text that represents a person's profile / skills / experience
- `job_title`   — the target label (what job they hold / are matched to)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# FILE-SPECIFIC COLUMN MAPPINGS for kandij/job-recommendation-datasets
# Each entry: (text_cols_to_combine, title_col)
# Multiple text columns are concatenated to form a richer text representation
# ──────────────────────────────────────────────────────────────────────────────

FILE_SCHEMA = {
    # Experience Data.csv  — typically has Job.Title + Skills + other fields
    'Experience Data.csv': {
        'text_cols' : ['Skills', 'Job.Title'],   # cols to concatenate for text
        'title_col' : 'Job.Title'
    },
    # jobs2.csv — job postings with title and description/skills
    'jobs2.csv': {
        'text_cols' : ['job_description', 'required_skills', 'title'],
        'title_col' : 'title'
    },
    # Salary Data.csv — salary + job title info
    'Salary Data.csv': {
        'text_cols' : ['Job Title', 'Education Level'],
        'title_col' : 'Job Title'
    },
    # combo_data.csv — combined dataset, usually richest
    'combo_data.csv': {
        'text_cols' : ['skills', 'position', 'experience'],
        'title_col' : 'position'
    },
}


def flexible_extract(df, text_cols_candidates, title_col_candidates):
    """
    Try each candidate column name (case-insensitive) and pick whichever exists.
    Returns (text_series, title_series) or (None, None) if nothing matched.
    """
    col_map = {c.lower().strip(): c for c in df.columns}

    # Resolve text columns
    found_text_cols = []
    for cand in text_cols_candidates:
        key = cand.lower().strip()
        if key in col_map:
            found_text_cols.append(col_map[key])

    # Resolve title column
    title_col = None
    for cand in (title_col_candidates if isinstance(title_col_candidates, list) else [title_col_candidates]):
        key = cand.lower().strip()
        if key in col_map:
            title_col = col_map[key]
            break

    if not found_text_cols or title_col is None:
        return None, None

    # Combine all found text columns into one string per row
    text_series = df[found_text_cols].fillna('').astype(str).agg(' '.join, axis=1)
    return text_series, df[title_col]


# Fallback auto-detect keywords if schema doesn't match
TEXT_KEYWORDS  = ['skill', 'resume', 'description', 'experience', 'summary',
                  'qualification', 'profile', 'responsibilities', 'about']
TITLE_KEYWORDS = ['title', 'role', 'position', 'job', 'category',
                  'designation', 'occupation']

def auto_detect(df):
    text_col, title_col = None, None
    for col in df.columns:
        cl = col.lower().replace(' ', '').replace('.', '').replace('_', '')
        if any(k in cl for k in TEXT_KEYWORDS) and text_col is None:
            text_col = col
        if any(k in cl for k in TITLE_KEYWORDS) and title_col is None:
            title_col = col
    return text_col, title_col


# ── Build unified DataFrame ────────────────────────────────────
segments = []

for fname, df in raw.items():
    if df is None:
        continue

    schema = FILE_SCHEMA.get(fname, {})
    text_s, title_s = flexible_extract(
        df,
        schema.get('text_cols', []),
        schema.get('title_col', [])
    )

    # Fallback to auto-detect if schema didn't match
    if text_s is None:
        print(f'  ⚠️  Schema mismatch for {fname} — auto-detecting columns...')
        tc, jc = auto_detect(df)
        if tc and jc:
            text_s  = df[tc].fillna('').astype(str)
            title_s = df[jc]
            print(f'     Auto-detected: text="{tc}" | title="{jc}"')
        else:
            print(f'     ❌ Could not extract from {fname} — skipping')
            print(f'     Available columns: {df.columns.tolist()}')
            continue

    seg = pd.DataFrame({
        'resume_text': text_s.values,
        'job_title'  : title_s.values,
        'source'     : fname
    })
    segments.append(seg)
    print(f'  ✅ Extracted from {fname}: {len(seg)} rows')

corpus = pd.concat(segments, ignore_index=True)
print(f'\n📊 Merged corpus shape: {corpus.shape}')
print(f'   Unique job titles  : {corpus["job_title"].nunique()}')
corpus.head(3)

## 🧹 Step 5: Clean & Preprocess Text

In [ ]:
def clean_text(text):
    """Normalize resume / job description text."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)   # remove URLs
    text = re.sub(r'\S+@\S+', ' ', text)             # remove emails
    text = re.sub(r'[^a-z\s]', ' ', text)            # keep letters only
    text = re.sub(r'\s+', ' ', text).strip()         # collapse whitespace
    return text

corpus = corpus.dropna(subset=['resume_text', 'job_title']).reset_index(drop=True)

corpus['cleaned_text'] = corpus['resume_text'].apply(clean_text)
corpus['job_title']    = corpus['job_title'].astype(str).str.strip().str.title()

# Drop rows where cleaned text is too short (< 4 words)
corpus = corpus[corpus['cleaned_text'].str.split().str.len() >= 4].reset_index(drop=True)

# Drop duplicate (text, title) pairs to avoid inflated similarity
corpus = corpus.drop_duplicates(subset=['cleaned_text', 'job_title']).reset_index(drop=True)

print(f'✅ Final corpus: {len(corpus)} rows | {corpus["job_title"].nunique()} unique job titles')
print(f'\nSample cleaned text (first row):')
print(corpus['cleaned_text'].iloc[0][:300])

## 📊 Step 6: EDA — Explore the Merged Dataset

In [ ]:
# ── Records per source file ────────────────────────────────────
print('Records per source file:')
print(corpus['source'].value_counts().to_string())

# ── Top Job Titles Bar Chart ───────────────────────────────────
top_n_chart = 20
top_titles = corpus['job_title'].value_counts().head(top_n_chart)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = cm.Blues_r(np.linspace(0.3, 0.9, top_n_chart))
top_titles.sort_values().plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title(f'Top {top_n_chart} Job Titles (merged corpus)', fontsize=13)
axes[0].set_xlabel('Count')
axes[0].tick_params(axis='y', labelsize=8)

# Text length distribution
corpus['word_count'] = corpus['cleaned_text'].str.split().str.len()
axes[1].hist(corpus['word_count'].clip(upper=300), bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('Text Length Distribution (words)', fontsize=13)
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'eda_plots.png'), dpi=120)
plt.show()

print(f'\nText length stats:')
print(corpus['word_count'].describe().round(1).to_string())

## 🧠 Step 7: TF-IDF Vectorization → Save PKL

In [ ]:
# ── TF-IDF Configuration ───────────────────────────────────────
tfidf = TfidfVectorizer(
    max_features = 15000,     # top 15K terms
    ngram_range  = (1, 2),    # unigrams + bigrams
    stop_words   = 'english', # remove common stop words
    sublinear_tf = True,      # apply log(1+tf) scaling
    min_df       = 2,         # ignore terms in < 2 documents
    max_df       = 0.90       # ignore terms in > 90% of documents
)

print('Fitting TF-IDF on merged corpus...')
tfidf_matrix = tfidf.fit_transform(corpus['cleaned_text'])

print(f'\n✅ TF-IDF matrix   : {tfidf_matrix.shape}  (docs × features)')
print(f'   Vocabulary size  : {len(tfidf.vocabulary_):,} terms')
print(f'   Matrix density   : {tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4%}')

# ── Save to /kaggle/working/models/ ───────────────────────────
PATHS = {
    'vectorizer' : os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'),
    'matrix'     : os.path.join(MODEL_DIR, 'tfidf_matrix.pkl'),
    'corpus'     : os.path.join(MODEL_DIR, 'corpus.pkl'),
}

with open(PATHS['vectorizer'], 'wb') as f: pickle.dump(tfidf, f)
with open(PATHS['matrix'],     'wb') as f: pickle.dump(tfidf_matrix, f)
with open(PATHS['corpus'],     'wb') as f:
    pickle.dump(corpus[['cleaned_text', 'job_title', 'source']].reset_index(drop=True), f)

print(f'\n💾 Saved PKL files:')
for k, p in PATHS.items():
    size_kb = os.path.getsize(p) / 1024
    print(f'   {k:12s} → {p}  ({size_kb:.0f} KB)')

## 📐 Step 8: Core Recommendation Function (Cosine Similarity)

In [ ]:
def recommend_jobs(resume_text: str, top_n: int = 5) -> pd.DataFrame:
    """
    Given raw resume text → return top_n recommended job titles
    using TF-IDF + Cosine Similarity against the merged corpus.

    Parameters
    ----------
    resume_text : str   — raw resume / skills text (any length)
    top_n       : int   — number of unique job title recommendations

    Returns
    -------
    pd.DataFrame with columns:
        Rank | Job Title | Avg Score | Best Score | Match Count
    """
    # ── Load saved models ──────────────────────────────────────
    with open(PATHS['vectorizer'], 'rb') as f: vec    = pickle.load(f)
    with open(PATHS['matrix'],     'rb') as f: matrix = pickle.load(f)
    with open(PATHS['corpus'],     'rb') as f: data   = pickle.load(f)

    # ── Clean + vectorize resume ───────────────────────────────
    cleaned = clean_text(resume_text)
    if len(cleaned.split()) < 3:
        return pd.DataFrame({'Message': ['Resume too short — please provide more text.']})

    resume_vec = vec.transform([cleaned])          # shape: (1, vocab)

    # ── Cosine similarity against all corpus docs ──────────────
    scores = cosine_similarity(resume_vec, matrix).flatten()  # shape: (n_docs,)

    # ── Aggregate scores by job title ─────────────────────────
    title_scores = defaultdict(list)
    for idx, score in enumerate(scores):
        if score < 0.01:          # ignore near-zero matches
            continue
        title = data.iloc[idx]['job_title']
        title_scores[title].append(score)

    if not title_scores:
        return pd.DataFrame({'Message': ['No matches found — try a more detailed resume.']})

    # ── Build results DataFrame ────────────────────────────────
    rows = []
    for title, sc_list in title_scores.items():
        rows.append({
            'Job Title'   : title,
            'Avg Score'   : round(np.mean(sc_list), 4),
            'Best Score'  : round(max(sc_list), 4),
            'Match Count' : len(sc_list)
        })

    results = (
        pd.DataFrame(rows)
        .sort_values('Avg Score', ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )
    results.index = results.index + 1           # rank starts at 1
    results.index.name = 'Rank'
    return results


print('✅ recommend_jobs() is ready!')

## 🧪 Step 9: Quick Smoke Test

In [ ]:
sample_resume = """
Experienced Python developer with 5 years in machine learning and data science.
Strong skills in scikit-learn, TensorFlow, Keras, pandas, NumPy, and SQL.
Built NLP pipelines for text classification, sentiment analysis, topic modelling.
Developed recommendation systems using collaborative and content-based filtering.
Familiar with A/B testing, statistical analysis, and data visualisation using Matplotlib and Seaborn.
Worked with AWS S3, EC2, deployed models with Flask REST APIs.
"""

results = recommend_jobs(sample_resume, top_n=10)
print('🎯 Top Job Recommendations:\n')
display(results.style.background_gradient(subset=['Avg Score'], cmap='Blues')
               .format({'Avg Score': '{:.4f}', 'Best Score': '{:.4f}'}))

best = results.iloc[0]['Job Title']
print(f'\n🏆  Best Match → {best}  (Avg Score: {results.iloc[0]["Avg Score"]:.4f})')

## 💼 Step 10: Test with Multiple Domain Resumes

In [ ]:
test_resumes = {
    'Finance / Accounting': """
        CPA with 8 years of experience in financial reporting, budgeting, forecasting.
        Proficient in Excel, SAP, QuickBooks. Strong knowledge of GAAP, IFRS.
        Managed audits, tax compliance, and cost analysis for mid-size firms.
        Skills: financial modelling, variance analysis, accounts payable and receivable.
    """,
    'Software Engineer': """
        Full-stack developer with expertise in Java, Spring Boot, React, Node.js.
        Experience building REST APIs, microservices, CI/CD pipelines using Jenkins.
        Worked on cloud infrastructure: Docker, Kubernetes, AWS, GCP.
        Strong in algorithms, data structures, system design, and code review.
    """,
    'Marketing / Sales': """
        Digital marketing specialist with 6 years in SEO, SEM, social media campaigns.
        Skilled in Google Analytics, HubSpot, Salesforce CRM.
        Led brand strategy, content creation, email marketing, and lead generation.
        Managed budgets over $500K, increased conversion rates by 35%.
    """,
    'Healthcare / Nursing': """
        Registered Nurse with 4 years in ICU and emergency care.
        Skills: patient assessment, IV therapy, medication administration, EHR systems.
        Experience with critical care, triage, wound management, patient education.
        BLS and ACLS certified, strong teamwork and communication skills.
    """
}

for domain, text in test_resumes.items():
    print(f'\n{'='*55}')
    print(f'🗂️  Domain: {domain}')
    print(f'{'='*55}')
    out = recommend_jobs(text, top_n=5)
    display(out.style.background_gradient(subset=['Avg Score'], cmap='Greens')
                     .format({'Avg Score': '{:.4f}', 'Best Score': '{:.4f}'}))

## 🖥️ Step 11: Interactive Interface — Paste Your Own Resume

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# ✏️  PASTE YOUR RESUME TEXT HERE  ↓
# ──────────────────────────────────────────────────────────────────────────────
MY_RESUME = """
Paste your resume text here...

Example:
Skilled project manager with PMP certification and 7 years of experience
managing cross-functional teams across IT and construction sectors.
Proficient in Jira, MS Project, Agile/Scrum, risk management, stakeholder
communication, and budget control up to $2M.
"""
# ──────────────────────────────────────────────────────────────────────────────
TOP_N = 10    # number of recommendations to show
# ──────────────────────────────────────────────────────────────────────────────

print('🔍 Analyzing resume...\n')
my_results = recommend_jobs(MY_RESUME, top_n=TOP_N)

if 'Message' in my_results.columns:
    print(my_results.iloc[0, 0])
else:
    display(
        my_results.style
            .background_gradient(subset=['Avg Score'], cmap='YlOrRd')
            .bar(subset=['Match Count'], color='#d4e6f1')
            .format({'Avg Score': '{:.4f}', 'Best Score': '{:.4f}'})
            .set_caption('🎯 Recommended Job Titles for Your Resume')
    )
    print(f'\n🏆 Top Recommendation → {my_results.iloc[0]["Job Title"]}')

## 📈 Step 12: Similarity Score Visualization

In [ ]:
def plot_recommendations(results, title='Resume → Job Match Scores'):
    if 'Message' in results.columns or 'Error' in results.columns:
        print('No results to plot.')
        return

    fig, ax = plt.subplots(figsize=(10, max(4, len(results) * 0.55)))
    colors  = cm.RdYlGn(np.linspace(0.3, 0.9, len(results)))

    bars = ax.barh(
        results['Job Title'][::-1],
        results['Avg Score'][::-1],
        color=colors[::-1],
        edgecolor='grey',
        linewidth=0.5
    )

    for bar, score in zip(bars, results['Avg Score'][::-1]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{score:.4f}', va='center', fontsize=9)

    ax.set_title(title, fontsize=14, fontweight='bold', pad=12)
    ax.set_xlabel('Cosine Similarity Score (Avg)', fontsize=11)
    ax.set_xlim(0, results['Avg Score'].max() * 1.25)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'recommendations.png'), dpi=130)
    plt.show()


# Plot for the sample DS resume
sample_results = recommend_jobs(sample_resume, top_n=10)
plot_recommendations(sample_results, title='Data Science Resume — Top Job Matches')

## 💾 Step 13: Reload Models (Run Independently Without Retraining)

In [ ]:
# Run this cell alone (after training) to reload and use the system
import pickle, os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

MODEL_DIR = '/kaggle/working/models/'
PATHS = {
    'vectorizer' : os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'),
    'matrix'     : os.path.join(MODEL_DIR, 'tfidf_matrix.pkl'),
    'corpus'     : os.path.join(MODEL_DIR, 'corpus.pkl'),
}

with open(PATHS['vectorizer'], 'rb') as f: tfidf        = pickle.load(f)
with open(PATHS['matrix'],     'rb') as f: tfidf_matrix = pickle.load(f)
with open(PATHS['corpus'],     'rb') as f: corpus       = pickle.load(f)

print('✅ Models reloaded!')
print(f'   Vocabulary  : {len(tfidf.vocabulary_):,} terms')
print(f'   Corpus rows : {len(corpus):,}')
print(f'   Job titles  : {corpus["job_title"].nunique()}')